In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

### Projektissa käytettävä datasetti ja sen lukeminen 

#### Tietoa datasetistä
Datasetti on peräisin [Kagglesta](https://www.kaggle.com/datasets/whisperingkahuna/premier-league-2324-team-and-player-insights/data), ja se tarjoaa kattavan tilastot jalkapallon Valioliigan kaudesta 2023/24, sisältäen dataa joukkueiden ja pelaajien suorituksista kaikilta ottelukierroksilta. Yhteensä datasetti koostuu yli 50 CSV-tiedostosta, joista jokainen keskittyy pelin tiettyihin osa-alueisiin, kuten joukkueiden suorituksiin, pelaajamittareihin, ottelutietoihin ja sarjataulukoihin.

#### Datasetin sisältö
Datasetti sisältää:
- **Joukkueiden suoritusmittarit**: Tarkat syötöt, päästetyt maalit, katkot jne.
- **Pelaajien suoritusmittarit**: Maaliodottama (xG), syötöt, voitetut taklaukset jne.
- **Ottelukohtaiset tiedot**: Tehdyt maalit, pallonhallintaprosentit, annetut kortit jne.
- **Sarjataulukot**: Sijoitukset, koti/vieras-suoritukset, ja edistyneet mittarit kuten xG ja xA.

#### Tiedostojen yksityiskohdat
Jokainen CSV-tiedosto sisältää tiettyyn aiheeseen liittyviä sarakkeita. Esimerkiksi:
- `player_top_scorers.csv`: Sijoitus, Pelaaja, Joukkue, Maalit, Maalit Rangaistuspotkuista, Minuutit, Ottelut, Kansalaisuus.
- `accurate_pass_team.csv`: Sijoitus, Joukkue, Onnistuneet syötöt per ottelu, Syöttöjen onnistumisprosentti (%), Ottelut, Kansalaisuus.
- `player_expected_goals.csv`: Sijoitus, Pelaaja, Joukkue, Maaliodottama (xG), Todelliset maalit, Minuutit, Ottelut.

TODO: Tarkka erittely datasetin sisältämistä tiedostoista ja niiden sisältämistä sarakkeista täällä:

#### Datan lukeminen

Kaggle tarjoaa useita vaihtoehtoisia tapoja datan tallentamiseen ja lukemiseen. Tämän projektin kohdalla päädyttiin viemään data GitHubiin, jotta vertaisarviointi onnistuu helposti (ts. notebookin ajaminen ei vaadi vertaisarvioijalta erillisiä toimenpiteitä).
Datasetti luetaan Pandas DataFrameen GitHubista analysointia varten.

Rakennetaan yksinkertainen funktio tiedostojen GitHubista lukemista varten:

In [ ]:
def readFile (filename, inMainDir = True):
    filename = filename.strip()
    if inMainDir:
        url = f"https://raw.githubusercontent.com/mikaelkankaanpaa/datatie2025_mk/main/project/data/Premleg_23_24/{filename}"
    else:
        url = f"https://raw.githubusercontent.com/mikaelkankaanpaa/datatie2025_mk/main/project/data/{filename}"
    try:
        df = pd.read_csv(url)
    except Exception as e:
        print(f"Dataa ei voitu lukea. Tarkista tiedoston nimi!\nVirheilmoitus: {e}")
        return None
    return df

Testataan lukea yksi datasetin tiedostoista (pelaajien tehdyt maalit; `player_top_scorers.csv`) dataFrameen, ja muodostetaan yleiskäsitys tiedoston sisällöstä:

In [ ]:
# topScorersDF = readFile("testi_vaara_nimi.csv") 
topScorersDF = readFile("player_top_scorers.csv")

if topScorersDF is not None:
    # dataFramen perustiedot
    display(topScorersDF.info())
    # ensimmäiset 5 riviä
    display(topScorersDF.head(5))

Datassa ei näytä olevan lainkaan NULL-arvoja, ja datatyypitkin vaikuttavat sopivilta. Datassa ei kuitenkaan ole omaa saraketta pelitilannemaaleille (ts. ei-pilkkumaaleille); lisätään se. Uudelleennimetään samalla 'Goals' -> 'Total Goals'

In [ ]:
topScorersDF.rename(columns={'Goals': 'Total Goals'}, inplace=True)

topScorersDF.insert(
    loc = 4, # yhteismaalien ja pilkkujen väliin
    column ='Non-Penalty Goals',
    value = topScorersDF['Total Goals'] - topScorersDF['Penalties']
)

Visualisoidaan top 10 maalintekijää dataFramen ja matplotlib-kirjaston tarjoamien visualisointityökalujen avulla. 
Lisätään 'peruskuvaajaan' pelaajakohtaiset rankkarimaalit, sekä selite jokaisen palkin perään, joka ilmaisee tarkan pelaajakohtaisen kokonaismaalimäärän, jotta kuvaaja on informatiivisempi:

(Matplotlibin värit: https://matplotlib.org/stable/gallery/color/named_colors.html)

In [ ]:
plt.figure(figsize=(10, 5))

# palkit 
barsGoals = (plt.barh(
    topScorersDF['Player'][:10], 
    topScorersDF['Total Goals'][:10], 
    label='Pelitilannemaalit',
    color='darkslateblue')
) 
barsPenalties = (plt.barh(
    topScorersDF['Player'][:10], topScorersDF['Penalties'][:10], 
    left=topScorersDF['Total Goals'][:10] - topScorersDF['Penalties'][:10], 
    label='Rangaistuspotkumaalit',
    color='mediumspringgreen')
)
# labelit ja otsikko
plt.xlabel("Maalit")
plt.ylabel("Pelaaja")
plt.title("Top 10 Maalintekijät Valioliigassa kaudella 2023/24")
plt.gca().invert_yaxis()

# selite maalien kokonaismäärästä 
for bar in barsGoals:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, 
             f'{int(bar.get_width())}', va='center')

plt.legend()
plt.show()


Matplotlib on toimiva, tuttu ja turvallinen. Testataan kuitenkin vielä jotain uutta: kokeillaan samalla datalla rakentaa interaktiivinen pylväskaavio [Plotly](https://plotly.com/python)-kirjaston avulla. (Tämä vielä keskeneräinen moduulin 4 palautuksessa)

(Plotly pylväskaaviot: https://plotly.com/python/bar-charts/)

In [ ]:
# Muutetaan ("sulatetaan") dataFrame wide formatista long formatiin, 
# jotta se sopii Plotly Expressin käyttöön
meltedTop10DF = topScorersDF[:10].melt(
    id_vars='Player', 
    value_vars=['Non-Penalty Goals', 'Penalties'], 
    var_name='Goal Type', 
    value_name='Goal Amount'
)

# meltedTop10DF['Goal Type'] = meltedTop10DF['Goal Type'].map(
#     {{'Non-Penalty Goals': 'Pelitilannemaalit', 'Penalties': 'Rangaistuspotkumaalit'}})

fig = px.bar(
    meltedTop10DF,
    x='Goal Amount',
    y='Player',
    color='Goal Type',
    orientation='h',
    labels={'Goal Amount': 'Maalit', 'Player': 'Pelaaja', 'Goal Type': 'Maalityyppi'},
    title="Top 10 Maalintekijät Valioliigassa kaudella 2023/24",
    color_discrete_map={'Non-Penalty Goals': 'darkslateblue', 'Penalties': 'mediumspringgreen'}
)
fig.update_layout(
    barmode='stack', # pinottu pylväsdiagrammi (toimii ilmankin)
    yaxis=dict(autorange="reversed") # käänteinen y:n järjestys
)
fig.show()

Kuvaajassa voi nyt esim. valita näkyviin haluamansa maalityypin (klikkaamalla maalityyppiä legendissä -> päälle/pois), hiiri palkin päälle viemällä tooltip näkyviin jne.